# 01 — Caso de negocio — Puntaje Saber 11 en Antioquia

Antes de abrir un solo dato, el proyecto necesita **una pregunta de negocio, una métrica que la refleje y un costo de equivocarse**.
Este notebook termina con esas tres cosas escritas **con números**, y su resumen va a `reports/00_caso.md`.

**Datos:** resultados oficiales del ICFES, prueba Saber 11, periodo 20224, colegios de Antioquia
(datos.gov.co, recurso `kgxf-xxbe`). Archivo: `data/raw/saber11_antioquia_2022.csv`; si no existe, se descarga
automáticamente de datos.gov.co la primera vez (no se sube a GitHub).

## 1. La pregunta de negocio

**Pregunta de negocio:** ¿Qué puntaje global en Saber 11 (`punt_global`) puede esperar un estudiante de Antioquia según las
características de su colegio, su perfil personal y su familia, y qué factores influyen más en ese puntaje?

**¿Quién decide y con qué información?** El resultado se entrega en una app web: el usuario ingresa el perfil de un estudiante
(variables del colegio `cole_`, del estudiante `estu_` y de la familia `fami_`) y obtiene su puntaje estimado. La usan:

| Usuario | Decisión que toma con el puntaje estimado |
|---|---|
| Secretaría de Educación | A qué colegios o municipios dirigir programas de refuerzo antes del examen |
| Rector / docentes | Qué estudiantes de grado 11 necesitan preparación adicional (preicfes, tutorías) |
| Estudiante / familia | A qué universidad o beca aspirar y si conviene invertir en preparación |

**¿Qué pasa hoy, sin el modelo?** El puntaje solo se conoce después de presentar la prueba, cuando ya no hay tiempo de actuar
para ese grupo de estudiantes.

## 2. Equivocarse no cuesta lo mismo

En este caso el grupo decidió que **los dos errores cuestan lo mismo**.

**Error más caro:** ninguno. Sobreestimar el puntaje (decir que le irá mejor de lo real) y subestimarlo (decir que le irá peor)
pesan igual.

**Error menos caro:** ninguno (ver arriba).

**Proporción entre los dos y de dónde sale:** 1 a 1. Equivocarse por 10 puntos hacia arriba cuesta lo mismo que equivocarse
por 10 puntos hacia abajo, y el costo crece en proporción a los puntos de error. Es decisión del grupo, porque la app la usan
perfiles distintos y ninguno de los dos errores es claramente más grave para todos.

In [1]:
import numpy as np
import pandas as pd

# Problema de regresión: no hay clases ni matriz de costos.
# Costo de equivocarse por un punto en cada dirección (decisión del grupo: 1 a 1).
COST_OVER = 1.0   # predice más puntaje del real (sobreestima)
COST_UNDER = 1.0  # predice menos puntaje del real (subestima)

pd.DataFrame(
    {"costo por punto de error": [COST_OVER, COST_UNDER]},
    index=["sobreestimar", "subestimar"],
)

,costo por punto de error
sobreestimar,1.0
subestimar,1.0


## 3. ¿Contra qué nos comparamos? (línea base)

Una métrica sola no dice nada: necesita un punto de referencia. Calculamos el error de **estrategias sin modelo**:
- **Siempre el promedio** de `punt_global`.
- **Siempre la mediana**, que es el valor constante con menor error absoluto promedio.
- **Azar según la distribución:** asignar a cada estudiante el puntaje de otro estudiante tomado al azar.

Cada fila de la fuente viene **duplicada exactamente** (144.766 filas = 72.383 estudiantes), así que la línea base se calcula
sin duplicados. El EDA confirmará estas cifras.

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))
from proyecto.data.load import load_raw_saber11

TARGET = "punt_global"
RANDOM_STATE = 42

# Si data/raw/saber11_antioquia_2022.csv no existe, se descarga de datos.gov.co la primera vez.
df = load_raw_saber11(ROOT / "data" / "raw")
print(f"Filas en el archivo: {len(df):,}")
df = df.drop_duplicates()
print(f"Filas sin duplicados exactos: {len(df):,}")

y = df[TARGET].to_numpy(dtype=float)
rng = np.random.default_rng(RANDOM_STATE)
baselines = {
    "Siempre el promedio": np.full_like(y, y.mean()),
    "Siempre la mediana": np.full_like(y, np.median(y)),
    "Azar según la distribución": rng.permutation(y),
}
pd.DataFrame(
    {
        "MAE (puntos)": {name: np.abs(y - pred).mean() for name, pred in baselines.items()},
        "RMSE (puntos)": {name: np.sqrt(((y - pred) ** 2).mean()) for name, pred in baselines.items()},
    }
).round(1)

Filas en el archivo: 144,766


Filas sin duplicados exactos: 72,383


,MAE (puntos),RMSE (puntos)
Siempre el promedio,42.9,51.6
Siempre la mediana,42.8,51.7
Azar según la distribución,58.9,72.9


**Lo que dice la línea base (una frase):** sin usar ninguna información del estudiante, predecir siempre la mediana
(242 puntos) se equivoca en promedio unos **42,8 puntos**; el modelo tiene que equivocarse claramente menos que eso para
aportar algo.

**¿La exactitud (accuracy) sirve como métrica en tu caso? ¿Por qué sí o por qué no?** No. La exactitud mide aciertos exactos
en clasificación; aquí predecimos un número (100 a 477 puntos) y casi nunca se acierta el valor exacto. Lo que importa es
**cuántos puntos** se equivoca el modelo.

In [3]:
# Línea base por segmento: la misma predicción constante (la mediana global, 242 puntos) evaluada dentro
# de cada grupo. Es solo la referencia sin modelo: no se entrena nada ni se parten los datos.
SEGMENT_COLS = {
    "fami_estratovivienda": "Estrato",
    "cole_area_ubicacion": "Zona",
    "cole_naturaleza": "Naturaleza del colegio",
}
median_global = np.median(y)
print(f"Mediana global: {median_global:.0f} puntos")
abs_error = (df[TARGET] - median_global).abs()

# Los vacíos se cuentan como un segmento propio, igual que «Sin Estrato».
baseline_by_segment = pd.concat(
    {
        label: abs_error.groupby(df[col].fillna("Sin dato (vacío)")).agg(
            **{"Estudiantes": "size", "MAE línea base (puntos)": "mean"}
        )
        for col, label in SEGMENT_COLS.items()
    },
    names=["Variable", "Segmento"],
)
baseline_by_segment.round(1)

Mediana global: 242 puntos


Estudiantes  MAE línea base (puntos)
Variable               Segmento                                              
Estrato                Estrato 1               14173                     38.4
                       Estrato 2               26652                     39.5
                       Estrato 3               18596                     44.4
                       Estrato 4                4317                     54.5
                       Estrato 5                1694                     65.8
                       Estrato 6                 668                     68.2
                       Sin Estrato              1561                     46.6
                       Sin dato (vacío)         4722                     44.6
Zona                   RURAL                   12539                     42.7
                       URBANO                  59844                     42.8
Naturaleza del colegio NO OFICIAL              14753                     53.4
                       OFICIAL                 57630                     40.1

**Lo que dice la línea base por segmento:** predecir siempre 242 puntos se equivoca mucho más en los estratos altos
(estrato 6: **68,2** puntos; estrato 5: **65,8**) y en los colegios no oficiales (**53,4**), porque sus puntajes están lejos
de la mediana; donde menos se equivoca es en el estrato 1 (**38,4**) y el estrato 2 (**39,5**). Entre zona rural (**42,7**)
y urbana (**42,8**) casi no hay diferencia. «Sin Estrato» (**46,6**) y los vacíos (**44,6**, 4.722 estudiantes) quedan por
encima del promedio. Ojo: en el estrato 1 la línea base (38,4) ya está casi en la meta mínima global de 38 puntos, así que
cumplir la meta global no garantiza la condición de equidad: hay que revisarla segmento por segmento.

## 4. Cómo se medirá el éxito

- **Métrica principal:** MAE (error absoluto medio, en puntos), *porque el grupo decidió que los dos errores cuestan lo mismo
  y que el costo crece en proporción a los puntos de error: eso es exactamente lo que mide el MAE. Además se explica sin
  formación técnica: "el modelo se equivoca en promedio ±N puntos".*
- **Métricas de control:** RMSE (revela si hay errores muy grandes que el MAE suaviza) y R² (qué parte de la variación del
  puntaje explica el modelo).
- **Qué resultado haría que el negocio diga «sirve»** (acordado por el grupo):
  - **Meta mínima: MAE de 38 puntos o menos** en validación cruzada agrupada por sede (`cole_cod_dane_sede`, `GroupKFold`),
    es decir, al menos un 11 % mejor que la línea base de 42,8 puntos (predecir siempre la mediana). *Motivo: el modelo solo
    aporta si se equivoca claramente menos que no usar ningún dato del estudiante; 38 puntos son 4,8 puntos menos de error en
    promedio. Se mide agrupando por sede porque así se mide el error con colegios que el modelo nunca vio, como pasará en la
    app (decisión 10 del EDA); partir por estudiante daría un MAE optimista.*
  - **Meta deseable: MAE de 35 puntos o menos** (un 18 % mejor que la línea base), medido de la misma forma. *Motivo: es el
    objetivo a buscar en el torneo y la optimización; no cumplirla no invalida el modelo si cumple la meta mínima.*
  - **Condición de equidad:** el MAE del modelo debe ser **menor que el MAE de la línea base dentro de cada segmento** de
    estrato (`fami_estratovivienda`, con «Sin Estrato» y los vacíos como segmentos propios), zona (`cole_area_ubicacion`) y
    naturaleza del colegio (`cole_naturaleza`). La línea base por segmento está en la tabla de la sección 3. *Motivo: un
    MAE promedio bueno puede esconder un grupo en el que el modelo se equivoca más que no usar modelo, y esas variables
    marcan a grupos que ya están en desventaja (sección 5). «Sin Estrato» y los vacíos van aparte porque tienen puntajes
    distintos al resto (decisión 6 del EDA).*

## 5. Lo que podría salir mal

- **Fuga de información (¿qué columna se conoce solo después de la decisión?):**
  - `punt_matematicas`, `punt_lectura_critica`, `punt_sociales_ciudadanas`, `punt_c_naturales`, `punt_ingles`: el puntaje
    global se calcula a partir de ellas.
  - `desemp_ingles`: nivel de desempeño en inglés, sale del resultado de la prueba.
  - `estu_estadoinvestigacion`: indica si el resultado se publicó o quedó en revisión; se conoce después del examen.
  - **Filas duplicadas:** si un mismo estudiante queda en entrenamiento y en prueba, la métrica sale optimista. Se eliminan
    los duplicados antes de partir.

  El modelo usará solo variables `cole_`, `estu_` y `fami_` (sin `estu_estadoinvestigacion`).
- **Sesgos o equidad (¿a quién podría perjudicar el modelo?):** variables como estrato, educación de los padres, municipio,
  zona rural/urbana, naturaleza del colegio o género pueden hacer que el modelo asigne puntajes bajos a grupos que ya están en
  desventaja. Se revisará el error por segmento en la etapa de validación.
- **Datos personales o confidenciales:** `estu_consecutivo` (identificador del estudiante: solo se usa para controlar
  duplicados, nunca como variable) y `estu_fechanacimiento`.
- **Hasta dónde puede decidir el modelo y dónde debe decidir una persona:** el puntaje estimado orienta (dónde reforzar, a qué
  aspirar), pero no debe usarse para excluir a un estudiante de un programa ni para etiquetarlo. La decisión final la toma una
  persona.

## 6. Cierre

- [x] Pregunta, quién decide y costos escritos **con números**.
- [x] Línea base calculada.
- [x] Métrica principal elegida y justificada.
- [x] Resumen copiado a `reports/00_caso.md`.
- [ ] Alguien que **no** escribió este notebook sabe explicar por qué los dos errores cuestan lo mismo.